# データの準備

In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
import matplotlib.pyplot as plt

In [4]:
import seaborn as sns
sns.set

<function seaborn.rcmod.set(*args, **kwargs)>

In [5]:
import tensorflow as tf

In [6]:
import tensorflow_datasets as tfds

C:\Users\09de1\Documents\python_basic\venv_python_basic\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

In [8]:
print(mnist_info)

tfds.core.DatasetInfo(
    name='mnist',
    full_name='mnist/3.0.1',
    description="""
    The MNIST database of handwritten digits.
    """,
    homepage='http://yann.lecun.com/exdb/mnist/',
    data_path='~\\tensorflow_datasets\\mnist\\3.0.1',
    file_format=tfrecord,
    download_size=11.06 MiB,
    dataset_size=21.00 MiB,
    features=FeaturesDict({
        'image': Image(shape=(28, 28, 1), dtype=tf.uint8),
        'label': ClassLabel(shape=(), dtype=tf.int64, num_classes=10),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    splits={
        'test': <SplitInfo num_examples=10000, num_shards=1>,
        'train': <SplitInfo num_examples=60000, num_shards=1>,
    },
    citation="""@article{lecun2010mnist,
      title={MNIST handwritten digit database},
      author={LeCun, Yann and Cortes, Corinna and Burges, CJ},
      journal={ATT Labs [Online]. Available: http://yann.lecun.com/exdb/mnist},
      volume={2},
      year={2010}
    }""",
)


In [9]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

In [10]:
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
print(num_validation_samples)

6000.0


In [11]:
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

In [12]:
num_test_samples = mnist_info.splits['test'].num_examples

In [14]:
num_test_samples = tf.cast(num_test_samples, tf.int64)

In [27]:
def scale(image, label):
    image = tf.cast(image, tf.float32)
    image /= 255.
    return image, label

In [28]:
scaled_train_and_validation_data = mnist_train.map(scale)

In [29]:
test_data = mnist_test.map(scale)

In [30]:
BUFFER_SIZE = 1000

In [31]:
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

In [32]:
validation_data = shuffled_train_and_validation_data.take(num_validation_samples)

In [33]:
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

In [34]:
BATCH_SIZE = 100

In [35]:
train_data = train_data.batch(BATCH_SIZE)

In [36]:
validation_data = validation_data.batch(num_validation_samples)

In [38]:
test_data = test_data.batch(num_test_samples)

In [39]:
validation_inputs, validation_targets = next(iter(validation_data))

# モデルの作成

In [40]:
# mnistのデータが28*28のため
input_size = 784

In [42]:
# mnistが0~9までの数字のため
output_size = 10

In [43]:
hidden_layer_size = 50

In [45]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28,1)),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'), # 一層目の隠れ層の指定
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

# 最適化アルゴリズムと損失関数の決定

In [47]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])
# sparse_categorical_crossentropy←onehotエンコーディングの時はこれ

# 訓練

In [49]:
# 繰り返しの回数
NUM_EPOCHS = 5

In [52]:
VALIDATION_STEPS = num_validation_samples

In [53]:
model.fit(train_data, epochs=NUM_EPOCHS, validation_data=(validation_inputs, validation_targets), validation_steps=VALIDATION_STEPS, verbose=2)

Epoch 1/5
540/540 - 12s - loss: 0.3993 - accuracy: 0.8852 - val_loss: 0.1967 - val_accuracy: 0.9435 - 12s/epoch - 23ms/step
Epoch 2/5
540/540 - 10s - loss: 0.1683 - accuracy: 0.9509 - val_loss: 0.1546 - val_accuracy: 0.9547 - 10s/epoch - 19ms/step
Epoch 3/5
540/540 - 10s - loss: 0.1282 - accuracy: 0.9621 - val_loss: 0.1301 - val_accuracy: 0.9593 - 10s/epoch - 19ms/step
Epoch 4/5
540/540 - 10s - loss: 0.1026 - accuracy: 0.9702 - val_loss: 0.1265 - val_accuracy: 0.9620 - 10s/epoch - 19ms/step
Epoch 5/5
540/540 - 11s - loss: 0.0876 - accuracy: 0.9746 - val_loss: 0.1109 - val_accuracy: 0.9680 - 11s/epoch - 20ms/step


In [54]:
test_loss, test_accuracy = model.evaluate(test_data)

1/1 [==============================] - 1s 887ms/step - loss: 0.1046 - accuracy: 0.9693


In [57]:
print('Test loss: {0:.2f}, Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

Test loss: 0.10, Test accuracy: 96.93%
